In [2]:
import os, sys, time
sys.path.append('..')
from dotenv import load_dotenv
import anthropic
from utils.helpers import CLAUDE_SONNET, CLAUDE_HAIKU, CLAUDE_OPUS, DEFAULT_MODEL

load_dotenv()
client = anthropic.Anthropic()


## Parte 1: Temperatura

In [8]:
if "client" not in dir():
    import sys; sys.path.append('..')
    from dotenv import load_dotenv
    import anthropic
    from utils.helpers import DEFAULT_MODEL
    load_dotenv()
    client = anthropic.Anthropic()

def ask_with_temperature(prompt: str, temperature: float, n: int = 3) -> list[str]:
    """Hace `n` peticiones con la misma temperatura para comparar variabilidad."""
    responses = []
    for _ in range(n):
        resp = client.messages.create(
            model=DEFAULT_MODEL,
            max_tokens=150,
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        responses.append(resp.content[0].text.strip())
    return responses

prompt = "Escribe un título creativo para una novela de ciencia ficción."

print("=== Temperature = 0.0 (Determinista) ===")
for r in ask_with_temperature(prompt, 0.0):
    print(f"  - {r}")

print("\n=== Temperature = 1.0 (Creativo) ===")
for r in ask_with_temperature(prompt, 1.0):
    print(f"  - {r}")


=== Temperature = 0.0 (Determinista) ===
  - # **Los Últimos Suspiros de Andrómeda**
  - # "Los Últimos Guardianes del Silencio Cuántico"

Este título evoca misterio, tecnología avanzada y una misión épica. Sugiere una historia donde quizás la comunicación o información cuántica es clave para la supervivencia, y un grupo selecto debe proteger un secreto cósmico.
  - # "Los Últimos Guardianes del Silencio Cuántico"

Este título evoca misterio, tecnología avanzada y una misión épica. Sugiere una historia donde quizás la comunicación o información cuántica es clave para la supervivencia, y un grupo selecto debe proteger un secreto cósmico.

=== Temperature = 1.0 (Creativo) ===
  - # **Los Últimos Suspiros de Andrómeda**
  - # "Los Últimos Guardianes del Silencio Cuántico"

Este título evoca misterio, tecnología avanzada y una misión épica. Sugiere una historia donde quizás la comunicación o información cuántica es clave para la supervivencia, y un grupo selecto debe proteger un secreto có

In [4]:
# Temperatura para tareas de código (baja temperatura recomendada)
print("=== Temperatura 0.0 para código ===")
resp = client.messages.create(
    model=DEFAULT_MODEL,
    max_tokens=300,
    temperature=0.0,
    messages=[{"role": "user", "content": 
               "Escribe una función Python que calcule el factorial de un número."}]
)
print(resp.content[0].text)


=== Temperatura 0.0 para código ===
# Función para calcular el factorial de un número

Aquí te muestro varias formas de calcular el factorial:

## 1. Versión iterativa (recomendada)
```python
def factorial(n):
    """
    Calcula el factorial de un número usando iteración.
    
    Args:
        n: Número entero no negativo
    
    Returns:
        El factorial de n
    """
    if n < 0:
        raise ValueError("El factorial no está definido para números negativos")
    
    resultado = 1
    for i in range(1, n + 1):
        resultado *= i
    
    return resultado
```

## 2. Versión recursiva
```python
def factorial_recursivo(n):
    """
    Calcula el factorial de un número usando recursión.
    
    Args:
        n: Número entero no negativo
    
    Returns:
        El factorial de n
    """
    if n < 0:
        raise ValueError("El factorial no está definido para números negativos")
    
    if n == 0 or n == 1:
        return 1
    
    return n * factorial_recursivo(n - 1)
`

## Parte 2: Response Streaming

In [9]:
import sys

print("=== Streaming con context manager ===")
print("Claude: ", end="", flush=True)

with client.messages.stream(
    model=DEFAULT_MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": "Explica qué es la inteligencia artificial en 3 párrafos."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print()  # Nueva línea al final


=== Streaming con context manager ===
Claude: ## ¿Qué es la Inteligencia Artificial?

La **inteligencia artificial (IA)** es un campo de la informática que bus ¿Qué es la Inteligencia Artificial?

La **inteligencia artificial (IA)** es un campo de la informática que busca crear sistemas y programas capaces de realizar tareas que normalmente requieren inteligencia humana. Estas tareas incluyen el reconca crear sistemas y programas capaces de realizar tareas que normalmente requieren inteligencia humana. Estas tareas incluyen el reconocimiento de voz, la toma de decisiones, la traducción de idiomas, el reconocimiento visualocimiento de voz, la toma de decisiones, la traducción de idiomas, el reconocimiento visual de objetos y el aprendizaje a partir de experiencias. En es de objetos y el aprendizaje a partir de experiencias. En esencia, se trata de enseñar a las máquinas a "pensar" y resolver problemas de manera autónoma.

Laencia, se trata de enseñar a las máquinas a "pensar" y resolver

In [6]:
# Streaming con eventos detallados
print("=== Streaming con eventos ===")
full_text = ""
token_count = 0

with client.messages.stream(
    model=DEFAULT_MODEL,
    max_tokens=200,
    messages=[{"role": "user", "content": "Escribe un haiku sobre la programacion."}]
) as stream:
    for event in stream:
        if hasattr(event, 'type'):
            if event.type == 'content_block_delta':
                if hasattr(event.delta, 'text'):
                    print(event.delta.text, end="", flush=True)
                    full_text += event.delta.text
                    token_count += 1

print(f"\n\nEstadisticas: ~{token_count} fragmentos recibidos")
print(f"Texto completo guardado: {len(full_text)} caracteres")


=== Streaming con eventos ===
CódigoCódigo en silencio,
errores flotan, debuggeo en silencio,
errores flotan, debuggeo—
compila al fin, paz.

Estadisticas: ~3 fragmentos recibidos
Texto completo guardado: 66 caracteres
—
compila al fin, paz.

Estadisticas: ~3 fragmentos recibidos
Texto completo guardado: 66 caracteres


In [7]:
# Comparar tiempo: streaming vs no-streaming
prompt = "Escribe un cuento corto de 100 palabras sobre un robot que aprende a cocinar."

# Sin streaming: espera hasta tener toda la respuesta
t0 = time.time()
resp = client.messages.create(
    model=DEFAULT_MODEL, max_tokens=300,
    messages=[{"role": "user", "content": prompt}]
)
t_no_stream = time.time() - t0
print(f"Sin streaming: {t_no_stream:.2f}s para recibir {resp.usage.output_tokens} tokens")

# Con streaming: primer token llega antes
t0 = time.time()
first_token_time = None
print("\nCon streaming: ", end="", flush=True)
with client.messages.stream(
    model=DEFAULT_MODEL, max_tokens=300,
    messages=[{"role": "user", "content": prompt}]
) as stream:
    for text in stream.text_stream:
        if first_token_time is None:
            first_token_time = time.time() - t0
        print(text, end="", flush=True)

print(f"\n\nTiempo al primer token: {first_token_time:.2f}s")


Sin streaming: 7.31s para recibir 227 tokens

Con streaming: ## El Chef de Circuitos

Roberto era un robot El Chef de Circuitos

Roberto era un robot de limpieza hasta que encontró un libro de recetas en de limpieza hasta que encontró un libro de recetas en el ático. Intrigado, decidió cocinar. el ático. Intrigado, decidió cocinar.

Su primer intento fue desastroso: confund

Su primer intento fue desastroso: confundió sal con azúcar y quemó el arroz. Peroió sal con azúcar y quemó el arroz. Pero Roberto no se rindió. Cada n Roberto no se rindió. Cada noche estudiaba, ajustaba sus sensores de temperatura y practicaba.oche estudiaba, ajustaba sus sensores de temperatura y practicaba.

Semanas después, su creador llegó exhausto del trabajo

Semanas después, su creador llegó exhausto del trabajo. Un aroma delicioso invadía la casa. En la mesa. Un aroma delicioso invadía la casa. En la mesa, una perfecta lasaña humeante, una perfecta lasaña humeante esperaba.

—¿Tú hiciste esto? esperaba.

—